# 13 — Informe reproducible previo al test

Genera tablas agregadas y una figura `ggplot2` para Sepsis-3 a seis horas usando exclusivamente development y validation. El test permanece cerrado. En el demo, todos los resultados son controles técnicos, no estimaciones clínicas.

**Entradas:** artefactos validados de los pasos 09–12 y configuración versionada. **Salidas:** tablas agregadas en memoria y `reports/figures/pretest_model_comparison.svg`. **Semilla:** `config/modeling.json`.

In [ ]:
from pathlib import Path
import json, shutil, subprocess, sys
import pandas as pd
from IPython.display import SVG, display
PROJECT_ROOT=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
sys.path.insert(0,str(PROJECT_ROOT/'src')) if str(PROJECT_ROOT/'src') not in sys.path else None
sys.path.insert(0,str(PROJECT_ROOT)) if str(PROJECT_ROOT) not in sys.path else None
from mimic_sepsis.artifacts import ArtifactStore, ArtifactValidationError
from mimic_sepsis.modeling import assemble_modeling_table
from mimic_sepsis.reporting import build_development_report
from scripts.build_demo_sofa_incremental import canonical_config
artifact_config=canonical_config()
model_config=json.loads((PROJECT_ROOT/'config/modeling.json').read_text())
evaluation_config=json.loads((PROJECT_ROOT/'config/evaluation.json').read_text())
valid=[]
for path in sorted((PROJECT_ROOT/'data/derived/sofa').glob('*/60_features/sepsis3_development_features.manifest.json')):
    try:
        manifest=ArtifactStore(path.parent).validate('sepsis3_development_features',expected_config=artifact_config); valid.append((manifest.created_at_utc,path.parents[1]))
    except (FileNotFoundError,ArtifactValidationError): pass
if not valid: raise RuntimeError('Ejecute primero los notebooks 00–12.')
RUN_ROOT=sorted(valid,key=lambda x:(x[0],str(x[1])))[-1][1]
landmarks=ArtifactStore(RUN_ROOT/'50_landmarks'); features=ArtifactStore(RUN_ROOT/'60_features')
print(f'Ejecución validada: {RUN_ROOT.name}')

## Muestras observadas y modelos comparables

In [ ]:
def load(partition):
    return assemble_modeling_table(
        landmarks.read_dataframe(f'sepsis3_{partition}_landmarks',expected_config=artifact_config),
        features.read_dataframe(f'sepsis3_{partition}_features',expected_config=artifact_config),
        horizon_hours=model_config['primary_horizon_hours'])
development=load('development'); validation=load('validation')
report=build_development_report(
    development,validation,feature_columns=model_config['clinical_baseline_features'],
    folds=model_config['cross_validation_folds'],logistic_c=model_config['logistic_c'],
    bootstrap_replicates=evaluation_config['bootstrap_replicates'],
    confidence_level=evaluation_config['confidence_level'],seed=model_config['seed'],
    exploratory_thresholds=evaluation_config['exploratory_threshold_probabilities'])
display(report.sample_flow)
display(report.point_metrics)
display(report.metric_intervals)
display(report.paired_intervals)
display(report.threshold_metrics)

## Comparación pareada con ggplot2

In [ ]:
rscript=shutil.which('Rscript')
if not rscript: raise RuntimeError('Rscript no está disponible en este kernel.')
figure_dir=PROJECT_ROOT/'reports/figures'; figure_dir.mkdir(parents=True,exist_ok=True)
csv=PROJECT_ROOT/'data/derived/pretest_intervals.csv'; dca_csv=PROJECT_ROOT/'data/derived/pretest_decision_curve.csv'; csv.parent.mkdir(parents=True,exist_ok=True)
svg=figure_dir/'pretest_model_comparison.svg'; dca_svg=figure_dir/'pretest_decision_curve.svg'; script=PROJECT_ROOT/'data/derived/pretest_plot.R'
report.paired_intervals.to_csv(csv,index=False)
report.decision_curves.to_csv(dca_csv,index=False)
script.write_text("""args <- commandArgs(trailingOnly=TRUE)
suppressPackageStartupMessages(library(ggplot2))
d <- read.csv(args[1]); d$metric <- factor(d$metric, levels=rev(unique(d$metric)))
p <- ggplot(d,aes(estimate,metric,colour=sample)) + geom_vline(xintercept=0,linetype=2,colour='grey50') + geom_errorbarh(aes(xmin=lower,xmax=upper),height=.18,position=position_dodge(width=.45)) + geom_point(position=position_dodge(width=.45),size=2) + labs(title='Baseline clínico frente a prevalencia',subtitle='Diferencias con IC percentil; bootstrap por paciente; test cerrado',x='Diferencia candidato menos referencia',y=NULL,colour='Muestra') + theme_minimal(base_size=11)
ggsave(args[2],p,width=10,height=5.5,device=grDevices::svg)
dca <- read.csv(args[3]); dca$strategy <- factor(dca$strategy,levels=c('candidate','reference','treat_all','treat_none'))
p2 <- ggplot(dca,aes(threshold,net_benefit,colour=strategy,linetype=strategy)) + geom_hline(yintercept=0,colour='grey70') + geom_line(linewidth=.8) + facet_wrap(~sample,scales='free_y') + labs(title='Decision-curve exploratoria',subtitle='Sin acción clínica ni umbral operativo aprobados; test cerrado',x='Umbral de probabilidad',y='Beneficio neto por landmark',colour='Estrategia',linetype='Estrategia') + theme_minimal(base_size=11)
ggsave(args[4],p2,width=10,height=5.5,device=grDevices::svg)
""",encoding='utf-8')
result=subprocess.run([rscript,str(script),str(csv),str(svg),str(dca_csv),str(dca_svg)],capture_output=True,text=True)
if result.returncode: raise RuntimeError(result.stderr)
display(SVG(filename=str(svg)))
display(SVG(filename=str(dca_svg)))

## Criterio de cierre

El informe es reproducible si los manifiestos validan, development/validation se ensamblan sin pérdidas, las réplicas agrupadas se contabilizan y ambas figuras se generan con `ggsave()`. La decision-curve es exploratoria porque todavía no existe acción clínica ni umbral operativo aprobado. No se selecciona un modelo por resultados del demo ni se abre el test. El informe final clínico requiere MIMIC-IV completo, decisiones firmadas y un único acceso registrado al test congelado.